# 05 — Measure Theory, Probability Theory & Their Connection to Statistics

> *A physicist's guide to the mathematical foundations that make probability rigorous.*

This notebook builds **measure theory from scratch** — the framework that Kolmogorov
used in 1933 to place probability on a solid axiomatic footing.  We connect every
abstract concept to concrete statistics on two familiar datasets:

| Dataset | Source | Rows | Purpose |
|---|---|---|---|
| **Titanic** | `seaborn` | 891 | Discrete / categorical probability |
| **California Housing** | `sklearn.datasets` | 20 640 | Continuous distributions & integrals |

Every section pairs **rigorous mathematical exposition** (with LaTeX) and **runnable code**
so you can verify each claim empirically.

---

## 1. Setup & Data Loading

We begin by importing the core scientific Python stack and loading both datasets.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.integrate import quad
from sklearn.datasets import fetch_california_housing
from itertools import chain, combinations

# Plotting defaults
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'font.size': 11,
    'figure.dpi': 100
})
sns.set_style("whitegrid")
np.random.seed(42)
print("✅ All imports successful.")

In [ ]:
# --- Titanic ---
titanic = sns.load_dataset('titanic').dropna(subset=['age', 'fare'])
print(f"Titanic: {titanic.shape[0]} rows × {titanic.shape[1]} cols")
print(titanic.head())

# --- California Housing ---
cal_raw = fetch_california_housing(as_frame=True)
housing = cal_raw.frame
print(f"\nCalifornia Housing: {housing.shape[0]} rows × {housing.shape[1]} cols")
print(housing.head())

---
## 2. Why Do We Need Measure Theory?

### 2.1 Three approaches to probability

Historically, probability has been formalised in three ways:

| Approach | Core idea | Limitation |
|---|---|---|
| **Classical** (Laplace) | $P(A) = \frac{\lvert A \rvert}{\lvert \Omega \rvert}$ — count equally-likely outcomes | Only works for finite, symmetric sample spaces |
| **Frequentist** (von Mises) | $P(A) = \lim_{n\to\infty} \frac{n_A}{n}$ — long-run relative frequency | Requires infinite repetition; limit may not exist |
| **Axiomatic** (Kolmogorov, 1933) | $P$ is a *measure* on a $\sigma$-algebra satisfying three axioms | Requires measure theory! |

Kolmogorov's approach is the one used in modern probability and statistics.
It unified the classical and frequentist views, and — crucially — it works for
**continuous** random variables, **infinite-dimensional** spaces (stochastic processes),
and every setting a physicist or data scientist encounters.

### 2.2 The problem: you can't measure everything

Naïvely, we'd like to assign a "length" or "probability" to *every* subset of $\mathbb{R}$.
But this leads to contradictions.

**The Vitali set (intuition, not full proof):**

1. Define an equivalence relation on $[0,1]$: $x \sim y$ iff $x - y \in \mathbb{Q}$.
2. Use the Axiom of Choice to pick exactly one representative from each class → a set $V$.
3. Translate $V$ by every rational $q \in \mathbb{Q} \cap [0,1]$ to get disjoint copies $V_q = V + q \pmod{1}$.
4. The copies tile $[0,1]$: $\; [0,1] = \bigsqcup_{q} V_q$.
5. If all copies had the same "length" $\ell$:
   - $\ell = 0 \;\Rightarrow\; 1 = \sum_q 0 = 0$  ✗
   - $\ell > 0 \;\Rightarrow\; 1 = \sum_q \ell = \infty$  ✗

**Conclusion:** there exist subsets of $\mathbb{R}$ that *cannot* be assigned a consistent
"length" compatible with translation-invariance and countable additivity.

### 2.3 The solution: σ-algebras

Instead of trying to measure *all* subsets, we restrict attention to a well-behaved
collection of subsets called a **$\sigma$-algebra** (Section 4).
The Vitali set is simply *not in* this collection — and we never miss it.

> 💡 **Physicist's Intuition:**  In quantum mechanics, not every Hermitian operator
> corresponds to a physically observable quantity — we restrict to *self-adjoint*
> operators on appropriate domains.  Similarly, measure theory restricts to
> *measurable* sets: the subsets about which we can meaningfully talk about size
> or probability.  The pathological sets we exclude are as unphysical as a
> non-self-adjoint "observable."

---
## 3. Sets Review

Before σ-algebras, we need fluency with set operations.

### 3.1 Basic operations

Let $\Omega$ be a universal set and $A, B \subseteq \Omega$.

| Operation | Notation | Definition |
|---|---|---|
| Union | $A \cup B$ | $\{x : x \in A \text{ or } x \in B\}$ |
| Intersection | $A \cap B$ | $\{x : x \in A \text{ and } x \in B\}$ |
| Complement | $A^c$ | $\{x \in \Omega : x \notin A\}$ |
| Difference | $A \setminus B$ | $A \cap B^c$ |
| Symmetric diff. | $A \triangle B$ | $(A \setminus B) \cup (B \setminus A)$ |

### 3.2 De Morgan's Laws

$$\left(\bigcup_i A_i\right)^c = \bigcap_i A_i^c, \qquad
  \left(\bigcap_i A_i\right)^c = \bigcup_i A_i^c$$

These are *essential* for proving that $\sigma$-algebras closed under union are
automatically closed under intersection.

### 3.3 Countable vs uncountable

- $\mathbb{N}, \mathbb{Z}, \mathbb{Q}$ are **countable** (can be listed $a_1, a_2, \ldots$).
- $\mathbb{R}, [0,1]$ are **uncountable** (Cantor's diagonal argument).
- Countable additivity works for countable unions; for *uncountable* unions it can fail.
  That's why $\sigma$-algebras demand closure under **countable** unions, not arbitrary ones.

In [ ]:
# --- Set operations in Python ---
Omega = set(range(1, 7))   # die: {1,2,3,4,5,6}
A = {1, 2, 3}              # "low roll"
B = {2, 4, 6}              # "even roll"

print(f"Ω       = {sorted(Omega)}")
print(f"A       = {sorted(A)}")
print(f"B       = {sorted(B)}")
print(f"A ∪ B   = {sorted(A | B)}")
print(f"A ∩ B   = {sorted(A & B)}")
print(f"Aᶜ      = {sorted(Omega - A)}")
print(f"A \\ B   = {sorted(A - B)}")
print(f"A △ B   = {sorted(A ^ B)}")

# De Morgan check
lhs = Omega - (A | B)
rhs = (Omega - A) & (Omega - B)
print(f"\nDe Morgan check:  (A∪B)ᶜ = Aᶜ∩Bᶜ ?  {lhs == rhs}  →  {sorted(lhs)}")

---
## 4. σ-Algebras

> 🔑 **Key Definition — σ-Algebra**
>
> A collection $\mathcal{F} \subseteq \mathcal{P}(\Omega)$ is a **σ-algebra** on $\Omega$ if:
>
> 1. $\Omega \in \mathcal{F}$  *(the whole space is measurable)*
> 2. $A \in \mathcal{F} \;\Rightarrow\; A^c \in \mathcal{F}$  *(closed under complementation)*
> 3. $A_1, A_2, \ldots \in \mathcal{F} \;\Rightarrow\; \bigcup_{i=1}^{\infty} A_i \in \mathcal{F}$
>    *(closed under **countable** union)*

**Immediate consequences** (from axioms + De Morgan):
- $\emptyset = \Omega^c \in \mathcal{F}$
- Closed under countable intersection: $\bigcap_i A_i = \left(\bigcup_i A_i^c\right)^c \in \mathcal{F}$
- Closed under set difference: $A \setminus B = A \cap B^c \in \mathcal{F}$

### 4.1 Canonical examples

| σ-algebra | On $\Omega$ | Contains |
|---|---|---|
| **Trivial** | any $\Omega$ | $\{\emptyset, \Omega\}$ — can only ask "did *something* happen?" |
| **Power set** $\mathcal{P}(\Omega)$ | any $\Omega$ | all $2^{|\Omega|}$ subsets — maximum information |
| **Coin flip** | $\{H,T\}$ | $\{\emptyset, \{H\}, \{T\}, \{H,T\}\} = \mathcal{P}(\{H,T\})$ |
| **Coarse die** | $\{1,\ldots,6\}$ | $\{\emptyset, \{1,2,3\}, \{4,5,6\}, \Omega\}$ — "low vs high" |
| **Fine die** | $\{1,\ldots,6\}$ | $\mathcal{P}(\{1,\ldots,6\})$ — all 64 subsets |

### 4.2 Generated σ-algebra

Given any collection of sets $\mathcal{C}$, the **generated σ-algebra**
$\sigma(\mathcal{C})$ is the *smallest* σ-algebra containing $\mathcal{C}$.
It's the intersection of all σ-algebras that contain $\mathcal{C}$.

### 4.3 Intuition: "the set of questions we can ask"

A σ-algebra encodes **available information**.  If you only know whether a die
roll is ≤ 3 or > 3, your σ-algebra is the coarse one above.  You *cannot* distinguish
individual faces — those events are not in your σ-algebra.

> 💡 **Physicist's Intuition:**  Think of a σ-algebra as the set of *observables*
> in a measurement apparatus.  A coarse detector tells you "particle went left or right."
> A fine detector tells you the exact angle.  The σ-algebra grows as your resolution improves.

### 4.4 Titanic σ-algebras

On the Titanic dataset, each choice of columns generates a different σ-algebra
(i.e., a different partition of passengers into distinguishable groups):

- **$\sigma(\{\text{survived}=1\})$**: can only ask "survived or not?" — 2 atoms
- **$\sigma(\{\text{sex}, \text{pclass}, \text{survived}\})$**: can ask about any
  combination of sex (2) × class (3) × survived (2) = 12 atoms

In [ ]:
# --- Build σ-algebras for a small sample space and verify axioms ---

def powerset(s):
    '''Return the power set of s as a set of frozensets.'''
    s = list(s)
    return set(
        frozenset(c) for c in chain.from_iterable(
            combinations(s, r) for r in range(len(s) + 1)
        )
    )

def is_sigma_algebra(F, Omega):
    '''Check whether F (a set of frozensets) is a sigma-algebra on Omega (a frozenset).'''
    # Axiom 1: Ω ∈ F
    if Omega not in F:
        return False, "Ω not in F"
    # Axiom 2: closed under complement
    for A in F:
        if Omega - A not in F:
            return False, f"complement of {set(A)} not in F"
    # Axiom 3 (finite check): closed under pairwise union (sufficient for finite F)
    for A in F:
        for B in F:
            if A | B not in F:
                return False, f"union {set(A)}∪{set(B)} not in F"
    return True, "all axioms satisfied"

# Die example
die = frozenset(range(1, 7))

# Trivial σ-algebra
trivial = {frozenset(), die}
print("Trivial σ-algebra:", is_sigma_algebra(trivial, die))

# Coarse σ-algebra: {∅, {1,2,3}, {4,5,6}, Ω}
low = frozenset({1, 2, 3})
high = frozenset({4, 5, 6})
coarse = {frozenset(), low, high, die}
print("Coarse σ-algebra: ", is_sigma_algebra(coarse, die))

# Power set (always a σ-algebra)
full = powerset(die)
print(f"Power set (|F|={len(full)}): ", is_sigma_algebra(full, die))

# A non-example: {∅, {1,2}, Ω}  — complement of {1,2} = {3,4,5,6} missing
bad = {frozenset(), frozenset({1, 2}), die}
print("Non-example:      ", is_sigma_algebra(bad, die))

In [ ]:
# --- Titanic: σ-algebra from {survived} vs {sex, pclass, survived} ---

# Atoms from survived only
atoms_surv = titanic.groupby('survived').ngroups
print(f"σ(survived):  {atoms_surv} atoms  →  2^{atoms_surv} = {2**atoms_surv} events")

# Atoms from (sex, pclass, survived)
atoms_full = titanic.groupby(['sex', 'pclass', 'survived']).ngroups
print(f"σ(sex, pclass, survived):  {atoms_full} atoms  →  2^{atoms_full} = {2**atoms_full} events")
print("\nAtom sizes:")
print(titanic.groupby(['sex', 'pclass', 'survived']).size().reset_index(name='count'))

---
## 5. The Borel σ-Algebra

### 5.1 Definition

The **Borel σ-algebra** on $\mathbb{R}$, denoted $\mathcal{B}(\mathbb{R})$, is the
σ-algebra *generated by all open intervals*:

$$\mathcal{B}(\mathbb{R}) \;=\; \sigma\!\bigl(\{(a,b) : a < b,\; a,b \in \mathbb{R}\}\bigr)$$

Equivalently, it's generated by half-lines $(-\infty, x]$ — which is why the **CDF**
$F(x) = P(X \le x)$ is so fundamental.

### 5.2 What's in $\mathcal{B}(\mathbb{R})$?

| Set type | In $\mathcal{B}(\mathbb{R})$? | Why |
|---|---|---|
| Open intervals $(a,b)$ | ✅ | generators |
| Closed intervals $[a,b]$ | ✅ | $[a,b] = \bigcap_{n=1}^\infty (a - 1/n,\; b + 1/n)$ |
| Singletons $\{x\}$ | ✅ | $\{x\} = \bigcap_{n=1}^\infty (x - 1/n,\; x + 1/n)$ |
| All open / closed sets | ✅ | open = countable union of open intervals |
| Countable sets ($\mathbb{Q}$, etc.) | ✅ | countable union of singletons |
| Vitali sets | ❌ | pathological; require Axiom of Choice |

### 5.3 Why this is enough for statistics

Every probability statement you'll encounter in statistics evaluates $P$ on a Borel set:

- $P(X \le x)$ → set $(-\infty, x]$
- $P(a < X < b)$ → set $(a, b)$
- $P(X = x)$ → set $\{x\}$
- $P(X \in A)$ for any interval, union of intervals, etc.

You will *never* need a non-Borel set in practice.  The Vitali set is a mathematical
curiosity — as unphysical as a non-measurable subset of phase space.

In [ ]:
# --- CDF and Borel sets: P(a < X < b) for California Housing MedInc ---
medinc = housing['MedInc'].values

a, b = 3.0, 5.0

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: empirical CDF
sorted_vals = np.sort(medinc)
ecdf = np.arange(1, len(sorted_vals) + 1) / len(sorted_vals)
axes[0].plot(sorted_vals, ecdf, linewidth=1.5)
axes[0].axvline(a, color='red', linestyle='--', alpha=0.7, label=f'a = {a}')
axes[0].axvline(b, color='red', linestyle='--', alpha=0.7, label=f'b = {b}')
Fa = np.searchsorted(sorted_vals, a) / len(sorted_vals)
Fb = np.searchsorted(sorted_vals, b) / len(sorted_vals)
axes[0].fill_between([a, b], [Fa, Fa], [Fb, Fb], alpha=0.3, color='orange',
                      label=f'P({a} < X < {b}) ≈ {Fb - Fa:.3f}')
axes[0].set_title('Empirical CDF — MedInc')
axes[0].set_xlabel('Median Income ($10k)')
axes[0].set_ylabel('F(x) = P(X ≤ x)')
axes[0].legend()

# Right: histogram with shaded region
axes[1].hist(medinc, bins=60, density=True, alpha=0.6, color='steelblue',
             edgecolor='white')
mask = (medinc > a) & (medinc < b)
axes[1].hist(medinc[mask], bins=30, density=True, alpha=0.8, color='orange',
             label=f'({a}, {b}) — Borel set')
axes[1].set_title(f'P({a} < MedInc < {b}) = {mask.mean():.3f}')
axes[1].set_xlabel('Median Income ($10k)')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Empirical P({a} < MedInc < {b}) = {mask.mean():.4f}")
print(f"From CDF:  F({b}) - F({a}) = {Fb:.4f} - {Fa:.4f} = {Fb - Fa:.4f}")

---
## 6. Measures

> 🔑 **Key Definition — Measure**
>
> A function $\mu : \mathcal{F} \to [0, \infty]$ is a **measure** on
> $(\Omega, \mathcal{F})$ if:
>
> 1. $\mu(\emptyset) = 0$
> 2. **Countable additivity:** for pairwise disjoint $A_1, A_2, \ldots \in \mathcal{F}$,
>    $$\mu\!\left(\bigsqcup_{i=1}^{\infty} A_i\right) = \sum_{i=1}^{\infty} \mu(A_i)$$

### 6.1 Important examples

| Measure | $\mu(A)$ | Domain |
|---|---|---|
| **Counting measure** | $\lvert A \rvert$ (number of elements) | any $\Omega$ |
| **Lebesgue measure** $\lambda$ | "length" — $\lambda([a,b]) = b - a$ | $(\mathbb{R}, \mathcal{B}(\mathbb{R}))$ |
| **Dirac measure** $\delta_x$ | $\delta_x(A) = \mathbf{1}_{x \in A}$ | any $\Omega$ |
| **Probability measure** $P$ | a measure with $P(\Omega) = 1$ | any $(\Omega, \mathcal{F})$ |

### 6.2 Key properties (following from the axioms)

- **Monotonicity:** $A \subseteq B \;\Rightarrow\; \mu(A) \le \mu(B)$
- **Subadditivity:** $\mu\!\left(\bigcup_i A_i\right) \le \sum_i \mu(A_i)$ (not necessarily disjoint)
- **Continuity from below:** $A_1 \subseteq A_2 \subseteq \cdots \;\Rightarrow\;
  \mu\!\left(\bigcup_i A_i\right) = \lim_{n\to\infty} \mu(A_n)$

In [ ]:
# --- Implement counting, Lebesgue, and Dirac measures ---

def counting_measure(A):
    '''Counting measure: mu(A) = |A| for any finite set A.'''
    return len(A)

def lebesgue_measure(interval):
    '''Lebesgue measure: mu([a,b]) = b - a.'''
    a, b = interval
    return b - a

def dirac_measure(x, A):
    '''Dirac measure: delta_x(A) = 1 if x in A else 0.'''
    return 1 if x in A else 0

# Counting measure
A = {1, 3, 5}
B = {2, 4}
print("=== Counting Measure ===")
print(f"μ(A)       = {counting_measure(A)}")
print(f"μ(B)       = {counting_measure(B)}")
print(f"μ(A ∪ B)   = {counting_measure(A | B)}")
print(f"μ(A)+μ(B)  = {counting_measure(A) + counting_measure(B)}")
print(f"Additivity (A∩B=∅): μ(A∪B) == μ(A)+μ(B) ? {counting_measure(A|B) == counting_measure(A)+counting_measure(B)}")

# Lebesgue measure
print("\n=== Lebesgue Measure ===")
print(f"λ([0, 1])   = {lebesgue_measure((0, 1))}")
print(f"λ([2, 5])   = {lebesgue_measure((2, 5))}")
print(f"λ([0,1])+λ([2,5]) = {lebesgue_measure((0,1)) + lebesgue_measure((2,5))}")
print(f"λ([0,1] ∪ [2,5]) = {lebesgue_measure((0,1)) + lebesgue_measure((2,5))}  (disjoint)")

# Dirac measure
print("\n=== Dirac Measure δ₃ ===")
print(f"δ₃({{1,2,3}}) = {dirac_measure(3, {1,2,3})}")
print(f"δ₃({{1,2}})   = {dirac_measure(3, {1,2})}")
print(f"δ₃({{3}})     = {dirac_measure(3, {3})}")
print(f"δ₃(∅)        = {dirac_measure(3, set())}")

In [ ]:
# --- Demonstrate monotonicity and subadditivity ---

print("=== Monotonicity ===")
A = {1, 2}
B = {1, 2, 3, 4, 5}
print(f"A = {sorted(A)},  B = {sorted(B)}")
print(f"A ⊆ B:  μ(A)={counting_measure(A)} ≤ μ(B)={counting_measure(B)}  ✓")

print("\n=== Subadditivity (non-disjoint sets) ===")
C = {1, 2, 3}
D = {2, 3, 4}
print(f"C = {sorted(C)},  D = {sorted(D)}")
print(f"μ(C ∪ D) = {counting_measure(C | D)}")
print(f"μ(C) + μ(D) = {counting_measure(C) + counting_measure(D)}")
print(f"μ(C ∪ D) ≤ μ(C) + μ(D):  {counting_measure(C|D)} ≤ {counting_measure(C)+counting_measure(D)}  ✓")
print(f"(Equality fails because C ∩ D = {sorted(C & D)} is non-empty)")

---
## 7. The Probability Space $(\Omega, \mathcal{F}, P)$

> 🔑 **Key Definition — Probability Space**
>
> A **probability space** is a triple $(\Omega, \mathcal{F}, P)$ where:
>
> - $\Omega$ is the **sample space** (set of all possible outcomes)
> - $\mathcal{F}$ is a **σ-algebra** on $\Omega$ (the measurable events)
> - $P : \mathcal{F} \to [0, 1]$ is a **probability measure**: a measure with $P(\Omega) = 1$

The **Kolmogorov axioms** are simply: $P$ is a measure (non-negative, countably additive)
with the normalisation $P(\Omega) = 1$.

### 7.1 Examples

| Experiment | $\Omega$ | $\mathcal{F}$ | $P$ |
|---|---|---|---|
| Fair coin | $\{H, T\}$ | $\mathcal{P}(\{H,T\})$ | $P(\{H\})=P(\{T\})=\tfrac{1}{2}$ |
| Fair die | $\{1,\ldots,6\}$ | $\mathcal{P}(\{1,\ldots,6\})$ | $P(\{k\})=\tfrac{1}{6}$ |
| Uniform on $[0,1]$ | $[0,1]$ | $\mathcal{B}([0,1])$ | $P([a,b]) = b - a$ (Lebesgue!) |
| Standard normal | $\mathbb{R}$ | $\mathcal{B}(\mathbb{R})$ | $P(A) = \int_A \frac{1}{\sqrt{2\pi}} e^{-x^2/2}\, dx$ |

### 7.2 Titanic as a probability space

- $\Omega$ = set of passengers (rows)
- $\mathcal{F}$ = σ-algebra generated by observable features
- $P(A) = \frac{|A|}{|\Omega|}$ = empirical (counting) measure, normalised

> 💡 **Physicist's Intuition:**  In classical mechanics, the probability space is
> (phase space $\Gamma$, Borel σ-algebra on $\Gamma$, Liouville measure $\rho\, dq\, dp$).
> The Liouville equation $\partial_t \rho = \{H, \rho\}$ governs how $P$ evolves.
> Kolmogorov's axioms are exactly the properties that make this work.

In [ ]:
# --- Construct probability spaces for Titanic ---
n = len(titanic)
print(f"Ω = set of {n} passengers (rows with non-null age & fare)")
print(f"|Ω| = {n}\n")

# P(survived=1)
p_surv = titanic['survived'].mean()
print(f"P(survived=1) = {titanic['survived'].sum()}/{n} = {p_surv:.4f}")

# P(sex='female' and pclass=1)
event = (titanic['sex'] == 'female') & (titanic['pclass'] == 1)
p_event = event.mean()
print(f"P(sex=female ∩ pclass=1) = {event.sum()}/{n} = {p_event:.4f}")

# Verify: P(Ω) = 1 and P(∅) = 0
print(f"\nP(Ω) = {n}/{n} = {n/n:.1f}  ✓")
print(f"P(∅) = 0/{n} = 0.0  ✓")

# Countable additivity: P(pclass=1) + P(pclass=2) + P(pclass=3) = 1
probs = titanic['pclass'].value_counts(normalize=True).sort_index()
print(f"\nP(pclass=1) = {probs.get(1, 0):.4f}")
print(f"P(pclass=2) = {probs.get(2, 0):.4f}")
print(f"P(pclass=3) = {probs.get(3, 0):.4f}")
print(f"Sum = {probs.sum():.4f}  (= P(Ω) ✓)")

In [ ]:
# --- Visualise the probability space partition ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: survived partition
surv_counts = titanic['survived'].value_counts().sort_index()
axes[0].bar(['Died (0)', 'Survived (1)'], surv_counts.values / n,
            color=['#d62728', '#2ca02c'], edgecolor='white')
axes[0].set_title('P from σ(survived)')
axes[0].set_ylabel('Probability')
for i, v in enumerate(surv_counts.values):
    axes[0].text(i, v/n + 0.01, f'{v/n:.3f}', ha='center', fontweight='bold')

# Right: (sex, pclass) partition
cross = titanic.groupby(['sex', 'pclass']).size() / n
cross = cross.reset_index(name='P')
cross['label'] = cross['sex'] + ', class ' + cross['pclass'].astype(str)
colors = sns.color_palette('Set2', len(cross))
axes[1].bar(cross['label'], cross['P'], color=colors, edgecolor='white')
axes[1].set_title('P from σ(sex, pclass)')
axes[1].set_ylabel('Probability')
axes[1].tick_params(axis='x', rotation=45)
for i, row in cross.iterrows():
    axes[1].text(i, row['P'] + 0.005, f"{row['P']:.3f}", ha='center', fontsize=9)

plt.tight_layout()
plt.show()

---
## 8. Random Variables as Measurable Functions

> 🔑 **Key Definition — Random Variable**
>
> A **random variable** is a *measurable function* $X : \Omega \to \mathbb{R}$.
>
> "Measurable" means: for every Borel set $B \in \mathcal{B}(\mathbb{R})$,
> the preimage $X^{-1}(B) = \{\omega \in \Omega : X(\omega) \in B\}$ belongs to $\mathcal{F}$.

### 8.1 It's a FUNCTION, not a "variable"

Despite the name, $X$ is a *function* that maps outcomes to numbers.
In the Titanic example:
- $X_{\text{age}} : \omega \mapsto \text{age of passenger } \omega$
- $X_{\text{fare}} : \omega \mapsto \text{fare paid by passenger } \omega$
- $X_{\text{survived}} : \omega \mapsto 1 \text{ if } \omega \text{ survived, else } 0$

The "randomness" comes from the probability measure $P$ on $\Omega$, not from $X$ itself.

### 8.2 CDF and the pushforward measure

The **CDF** is:
$$F_X(x) = P(X \le x) = P\bigl(\{\omega \in \Omega : X(\omega) \le x\}\bigr)$$

This defines the **pushforward** (or **induced**) measure $P_X$ on $(\mathbb{R}, \mathcal{B}(\mathbb{R}))$:
$$P_X(B) = P(X^{-1}(B)) = P(\{\omega : X(\omega) \in B\})$$

The pushforward $P_X$ is the *distribution* of $X$.

### 8.3 Discrete vs continuous

- **Discrete:** $X$ takes countably many values.  $P(X = x) > 0$ for each value.
- **Continuous:** $X$ has a density $f_X$.  Crucially, $P(X = x) = 0$ for every
  individual $x$ — probability lives on *intervals*, not *points*.

This is a direct consequence of the Lebesgue measure: $\lambda(\{x\}) = 0$.

In [ ]:
# --- Random variables as functions on Titanic ---

# Define "random variables" as functions from passenger (row) to ℝ
X_age = titanic['age'].values
X_fare = titanic['fare'].values
X_survived = titanic['survived'].values

print("=== Random variable X_age : Ω → ℝ ===")
print(f"X_age(ω₀) = {X_age[0]}")
print(f"X_age(ω₁) = {X_age[1]}")
print(f"X_age(ω₂) = {X_age[2]}")
print()

# P(X_age > 50): compute BOTH as direct count and via the pushforward
event_direct = np.mean(X_age > 50)
print(f"P(age > 50) = count(age>50) / |Ω| = {np.sum(X_age > 50)}/{len(X_age)} = {event_direct:.4f}")

# Via CDF: P(age > 50) = 1 - F(50)
F50 = np.mean(X_age <= 50)
print(f"Via CDF:     1 - F(50) = 1 - {F50:.4f} = {1 - F50:.4f}")
print(f"Match: {np.isclose(event_direct, 1 - F50)}")

In [ ]:
# --- CDF and pushforward visualisation ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Discrete RV: survived (Bernoulli)
vals, counts = np.unique(X_survived, return_counts=True)
probs = counts / len(X_survived)
axes[0].bar(vals, probs, color=['#d62728', '#2ca02c'], edgecolor='white', width=0.4)
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['0 (died)', '1 (survived)'])
axes[0].set_title('Pushforward $P_{X_{survived}}$ (discrete)')
axes[0].set_ylabel('P(X = x)')
for i, (v, p) in enumerate(zip(vals, probs)):
    axes[0].text(v, p + 0.01, f'{p:.3f}', ha='center')

# 2. Continuous RV: age — histogram ≈ density
axes[1].hist(X_age, bins=40, density=True, alpha=0.7, color='steelblue',
             edgecolor='white')
axes[1].set_title('Pushforward $P_{X_{age}}$ (continuous)')
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Density f(x)')
axes[1].axvline(50, color='red', linestyle='--', label='x = 50')
axes[1].legend()

# 3. Empirical CDF of age
sorted_age = np.sort(X_age)
ecdf_age = np.arange(1, len(sorted_age) + 1) / len(sorted_age)
axes[2].plot(sorted_age, ecdf_age, linewidth=1.5, color='steelblue')
axes[2].axhline(F50, color='grey', linestyle=':', alpha=0.6)
axes[2].axvline(50, color='red', linestyle='--', alpha=0.6)
axes[2].plot(50, F50, 'ro', markersize=8)
axes[2].set_title(f'CDF: $F_{{age}}(50) = {F50:.3f}$')
axes[2].set_xlabel('Age')
axes[2].set_ylabel('F(x) = P(X ≤ x)')

plt.tight_layout()
plt.show()

### 8.4 Key point: P(X = x) = 0 for continuous random variables

For continuous $X$ with density $f$:
$$P(X = x) = P_X(\{x\}) = \int_{\{x\}} f(t)\,dt = 0$$

because the Lebesgue measure of a single point is zero: $\lambda(\{x\}) = 0$.

This is why we always compute $P(a < X < b)$, never $P(X = x)$, for continuous
distributions.  The probability lives on *intervals* (Borel sets with positive
Lebesgue measure), not on individual points.

In [ ]:
# --- Demonstrate P(X = x) ≈ 0 for continuous RV ---
from scipy.stats import norm

mu, sigma = X_age.mean(), X_age.std()
fitted = norm(loc=mu, scale=sigma)

x_point = 30.0
dx = 0.001

# P(X = 30) ≈ 0 (point probability)
p_point = fitted.cdf(x_point + dx/2) - fitted.cdf(x_point - dx/2)
print(f"P(X = {x_point}) ≈ P({x_point - dx/2} < X < {x_point + dx/2}) = {p_point:.6f}")
print(f"  → as dx → 0, this → 0")

# P(29 < X < 31) = meaningful interval probability
p_interval = fitted.cdf(31) - fitted.cdf(29)
print(f"\nP(29 < X < 31) = {p_interval:.4f}  ← meaningful!")

# Empirical check
p_emp = np.mean((X_age > 29) & (X_age < 31))
print(f"Empirical P(29 < age < 31) = {p_emp:.4f}")

---
## 9. The Lebesgue Integral

The Lebesgue integral is the *right* notion of integration for measure theory.
It generalises the Riemann integral and — most importantly for us — it gives the
rigorous definition of **expectation**.

### 9.1 Riemann vs Lebesgue: two philosophies

**Riemann integral** — partition the **domain**:

$$\int_a^b f(x)\,dx \approx \sum_{i} f(x_i^*)\,\Delta x_i$$

Slice the $x$-axis into small intervals, multiply each by the function height.
Works for continuous functions, but fails for "wild" functions like the Dirichlet
function $\mathbf{1}_\mathbb{Q}$ (1 on rationals, 0 on irrationals).

**Lebesgue integral** — partition the **range**:

$$\int f\,d\mu \approx \sum_{j} y_j \cdot \mu\bigl(\{x : f(x) \approx y_j\}\bigr)$$

Slice the $y$-axis.  For each output value $y_j$, measure the *size* of the
set of inputs mapping there.  Then sum $y_j \times \text{size}$.

> 💡 **"Coin sorting" analogy:**
> - Riemann: count your coins in the order they sit on the table.
> - Lebesgue: sort coins by denomination first, then count each pile.
> Same total, but sorting makes it *possible* to handle messier collections.

### 9.2 Formal construction

**Step 1: Simple functions.**  A *simple function* is a finite sum
$$\varphi = \sum_{i=1}^n a_i \,\mathbf{1}_{A_i}, \qquad A_i \in \mathcal{F},\; a_i \ge 0$$

Its integral is:
$$\int \varphi \; d\mu = \sum_{i=1}^{n} a_i \cdot \mu(A_i)$$

**Step 2: General non-negative measurable functions.**
Approximate $f \ge 0$ from below by an increasing sequence of simple functions
$\varphi_n \uparrow f$, then define:
$$\int f \; d\mu = \lim_{n \to \infty} \int \varphi_n \; d\mu$$

**Step 3: General measurable functions.**
Write $f = f^+ - f^-$ where $f^+ = \max(f, 0)$ and $f^- = \max(-f, 0)$,
then $\int f\, d\mu = \int f^+\, d\mu - \int f^-\, d\mu$ (provided at least one is finite).

### 9.3 Why Lebesgue is better

1. **More functions are integrable**: Dirichlet function has $\int \mathbf{1}_\mathbb{Q}\, d\lambda = 0$ (rationals have measure zero).
2. **Convergence theorems**: Monotone Convergence, Dominated Convergence, Fatou's Lemma — essential tools.
3. **Unifies sums and integrals**:
   - Counting measure: $\int f\, d\mu_{\text{count}} = \sum_i f(x_i)$
   - Lebesgue measure: $\int f\, d\lambda = \int_a^b f(x)\, dx$
   - Any measure works the same way!

### 9.4 THE CONNECTION: Expectation = Lebesgue integral

$$\boxed{E[X] = \int_\Omega X \; dP}$$

For continuous $X$ with density $f$:
$$E[X] = \int_\Omega X\, dP = \int_{-\infty}^{\infty} x \, f(x)\, dx$$

For discrete $X$:
$$E[X] = \int_\Omega X\, dP = \sum_x x \cdot P(X = x)$$

Both are special cases of the same Lebesgue integral — just with different
underlying measures!

> 💡 **Physicist's Intuition:**  The Lebesgue integral is to the Riemann integral
> what Feynman's **path integral** is to classical action evaluation.  In the path
> integral, you "partition by action values" — grouping all paths with action ≈ $S$,
> weighting by $e^{iS/\hbar}$, and summing.  The Lebesgue integral partitions by
> *output values* and measures the preimage — same philosophy, different context.

In [ ]:
# --- Riemann vs Lebesgue approach: numerical demonstration ---

def f(x):
    '''Example function: sin(x) * exp(-x/5) on [0, 10]'''
    return np.sin(x) * np.exp(-x / 5)

a_int, b_int = 0, 10

# --- Riemann approach: partition the DOMAIN ---
n_riemann = 1000
x_r = np.linspace(a_int, b_int, n_riemann + 1)
dx = (b_int - a_int) / n_riemann
riemann_sum = np.sum(f((x_r[:-1] + x_r[1:]) / 2) * dx)

# --- Lebesgue-style approach: partition the RANGE ---
n_eval = 100000
x_fine = np.linspace(a_int, b_int, n_eval)
y_fine = f(x_fine)
y_min, y_max = y_fine.min(), y_fine.max()

n_levels = 200
levels = np.linspace(y_min, y_max, n_levels + 1)
lebesgue_sum = 0.0
dx_fine = (b_int - a_int) / n_eval

for i in range(n_levels):
    y_lo, y_hi = levels[i], levels[i + 1]
    y_mid = (y_lo + y_hi) / 2
    # "Measure" of preimage: total length of x where f(x) ∈ [y_lo, y_hi)
    mask = (y_fine >= y_lo) & (y_fine < y_hi)
    preimage_measure = mask.sum() * dx_fine
    lebesgue_sum += y_mid * preimage_measure

# Scipy reference
from scipy.integrate import quad
exact, _ = quad(f, a_int, b_int)

print("=== Riemann vs Lebesgue Integration ===")
print(f"∫₀¹⁰ sin(x)·exp(-x/5) dx")
print(f"  Riemann (n={n_riemann}):     {riemann_sum:.6f}")
print(f"  Lebesgue (levels={n_levels}): {lebesgue_sum:.6f}")
print(f"  Exact (scipy.quad):          {exact:.6f}")

In [ ]:
# --- Visualise the two approaches ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
x_plot = np.linspace(a_int, b_int, 500)
y_plot = f(x_plot)

# Left: Riemann — partition domain
n_vis = 20
x_bars = np.linspace(a_int, b_int, n_vis + 1)
for i in range(n_vis):
    x_mid = (x_bars[i] + x_bars[i+1]) / 2
    height = f(x_mid)
    color = '#2ca02c' if height >= 0 else '#d62728'
    axes[0].bar(x_mid, height, width=(x_bars[i+1]-x_bars[i]),
                alpha=0.4, color=color, edgecolor='grey', linewidth=0.5)
axes[0].plot(x_plot, y_plot, 'k-', linewidth=2)
axes[0].set_title('Riemann: partition the DOMAIN (x-axis)')
axes[0].set_xlabel('x')
axes[0].set_ylabel('f(x)')
axes[0].axhline(0, color='grey', linewidth=0.5)

# Right: Lebesgue — partition range, colour by level sets
n_lev_vis = 10
levels_vis = np.linspace(y_min, y_max, n_lev_vis + 1)
cmap = plt.cm.coolwarm
for i in range(n_lev_vis):
    y_lo, y_hi = levels_vis[i], levels_vis[i+1]
    mask_vis = (y_plot >= y_lo) & (y_plot < y_hi)
    color_val = (i + 0.5) / n_lev_vis
    axes[1].scatter(x_plot[mask_vis], y_plot[mask_vis], c=[cmap(color_val)],
                    s=3, alpha=0.7)
    # Draw horizontal band
    axes[1].axhspan(y_lo, y_hi, alpha=0.1, color=cmap(color_val))
axes[1].plot(x_plot, y_plot, 'k-', linewidth=1.5, alpha=0.3)
axes[1].set_title('Lebesgue: partition the RANGE (y-axis)')
axes[1].set_xlabel('x')
axes[1].set_ylabel('f(x)')
axes[1].axhline(0, color='grey', linewidth=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# --- E[X] as a Lebesgue integral: Titanic age ---

# Empirical expectation: E[X] = (1/n) Σ x_i  (integral w.r.t. empirical measure)
E_empirical = np.mean(X_age)

# Fit a normal and compute E[X] = ∫ x f(x) dx  (integral w.r.t. Lebesgue measure)
mu_fit, sigma_fit = norm.fit(X_age)
E_analytical = mu_fit   # For normal, E[X] = μ

# Numerical integration: ∫ x · f(x) dx
E_numerical, _ = quad(lambda x: x * norm.pdf(x, mu_fit, sigma_fit), 0, 100)

print("=== E[X] = ∫ X dP  (age) ===")
print(f"Empirical (sum/n):          E[age] = {E_empirical:.4f}")
print(f"Fitted normal μ:            E[age] = {mu_fit:.4f}")
print(f"Numerical ∫ x·f(x)dx:      E[age] = {E_numerical:.4f}")
print()

# Discrete example: E[survived]
# E[X] = Σ x·P(X=x) = 0·P(X=0) + 1·P(X=1) = P(X=1)
E_surv = np.mean(X_survived)
print("=== E[X] = Σ x·P(X=x)  (survived, discrete) ===")
print(f"E[survived] = 0·P(0) + 1·P(1) = P(survived=1) = {E_surv:.4f}")
print(f"Direct mean:                                     {np.mean(X_survived):.4f}")

In [ ]:
# --- Visualise E[X] as area under x·f(x) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x_range = np.linspace(0, 80, 500)
pdf_vals = norm.pdf(x_range, mu_fit, sigma_fit)
integrand = x_range * pdf_vals

# Left: the density f(x)
axes[0].plot(x_range, pdf_vals, 'steelblue', linewidth=2)
axes[0].fill_between(x_range, pdf_vals, alpha=0.3, color='steelblue')
axes[0].axvline(E_empirical, color='red', linestyle='--', linewidth=2,
                 label=f'E[X] = {E_empirical:.1f}')
axes[0].set_title('Density $f_X(x)$')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('$f(x)$')
axes[0].legend()

# Right: the integrand x·f(x) — E[X] is the area under this curve
axes[1].plot(x_range, integrand, 'darkorange', linewidth=2)
axes[1].fill_between(x_range, integrand, alpha=0.3, color='orange')
axes[1].axvline(E_empirical, color='red', linestyle='--', linewidth=2,
                 label=f'E[X] = ∫ x·f(x)dx = {E_numerical:.1f}')
axes[1].set_title('Integrand $x \\cdot f_X(x)$  — area = $E[X]$')
axes[1].set_xlabel('Age')
axes[1].set_ylabel('$x \\cdot f(x)$')
axes[1].legend()

plt.tight_layout()
plt.show()

# 10. Integration with Respect to a Measure

## The Grand Unification of Sums, Integrals, and Expectations

The Lebesgue integral $\int f\,d\mu$ **unifies** every "adding-up" operation you've ever seen:

| Measure $\mu$ | $\int f\,d\mu$ becomes | Familiar form |
|---|---|---|
| Counting measure on $\{x_1,\dots,x_n\}$ | $\displaystyle\sum_i f(x_i)$ | **Summation** |
| Lebesgue measure on $\mathbb{R}$ | $\displaystyle\int f(x)\,dx$ | **Ordinary integral** |
| Probability measure $P$ | $E[f(X)]$ | **Expectation** |
| Empirical measure $\hat P_n = \frac{1}{n}\sum_{i=1}^n \delta_{x_i}$ | $\frac{1}{n}\sum_{i=1}^n f(x_i)$ | **Sample average** |

> 💡 **Physicist's Intuition:** This is exactly like writing $\langle \psi | \hat{A} | \psi \rangle$
> in different bases — the *operator* is the same, only the *representation* changes.

**Key idea:** choose the measure, and the integral adapts automatically.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, integrate

# --- Demonstration: same function, different measures ---
# f(x) = x^2

# 1) Counting measure on {1,2,3,4,5}: sum of f(x_i)
points = np.array([1, 2, 3, 4, 5])
counting = np.sum(points**2)  # Σ f(x_i)
print("=== Integration w.r.t. Different Measures ===\n")
print(f"f(x) = x²  on {{1,2,3,4,5}}")
print(f"  Counting measure  → Σ f(xᵢ)       = {counting}")

# 2) Lebesgue measure on [1,5]: ordinary integral
lebesgue_val, _ = integrate.quad(lambda x: x**2, 1, 5)
print(f"  Lebesgue measure  → ∫₁⁵ x² dx     = {lebesgue_val:.4f}")

# 3) Probability measure (uniform on [1,5]): expectation
# density = 1/4 on [1,5]
expectation_val, _ = integrate.quad(lambda x: x**2 * (1/4), 1, 5)
print(f"  Uniform prob meas → E[X²]          = {expectation_val:.4f}")

# 4) Empirical measure from data: sample average
np.random.seed(42)
sample = np.random.uniform(1, 5, size=10000)
empirical_val = np.mean(sample**2)
print(f"  Empirical measure → (1/n)Σf(xᵢ)   = {empirical_val:.4f}")
print(f"  (should ≈ E[X²] = {expectation_val:.4f})")


In [ ]:
# --- Apply to Titanic and Housing data ---
print("=== Expectations as Integrals on Real Data ===\n")

# Titanic: E[fare] via empirical measure = sample mean
fares = titanic['fare'].dropna()
emp_mean_fare = fares.mean()  # = ∫ fare d(empirical measure)
print(f"Titanic: E[fare] via empirical measure = {emp_mean_fare:.2f}")

# Counting measure on survived: total survivors
surv_count = titanic['survived'].sum()  # ∫ survived d(counting)
print(f"Titanic: ∫ survived d(counting) = Σ survived = {surv_count}")

# Probability measure: P(survived) = E[survived]
p_surv = titanic['survived'].mean()  # ∫ survived dP
print(f"Titanic: E[survived] = ∫ survived dP = {p_surv:.4f}")

# Housing: E[MedHouseVal] and E[MedHouseVal²]
med_vals = housing['MedHouseVal']
e_x = med_vals.mean()
e_x2 = (med_vals**2).mean()
var_x = e_x2 - e_x**2
print(f"\nHousing: E[MedHouseVal]   = {e_x:.4f}")
print(f"Housing: E[MedHouseVal²]  = {e_x2:.4f}")
print(f"Housing: Var = E[X²]-E[X]² = {var_x:.4f}  (check: {med_vals.var():.4f})")


# 11. Convergence Theorems — When Can You Swap Limits and Integrals?

These three theorems are the **engine room** of analysis and probability.

## Monotone Convergence Theorem (MCT)

If $0 \le f_1 \le f_2 \le \cdots$ and $f_n \uparrow f$ pointwise, then

$$\lim_{n\to\infty} \int f_n\,d\mu = \int f\,d\mu$$

*Increasing, non-negative sequences? Swap limit and integral freely.*

## Dominated Convergence Theorem (DCT)

If $f_n \to f$ pointwise, and $|f_n| \le g$ for all $n$ with $\int g\,d\mu < \infty$, then

$$\lim_{n\to\infty} \int f_n\,d\mu = \int f\,d\mu$$

*Bounded by an integrable function? Swap limit and integral freely.*

## Fatou's Lemma

For non-negative $f_n$:

$$\int \liminf_{n\to\infty} f_n\,d\mu \;\le\; \liminf_{n\to\infty} \int f_n\,d\mu$$

*Without extra conditions, you only get an inequality.*

### Why they matter

- **LLN proof** uses DCT (bounded case) or MCT (truncation argument)
- **CLT proof** uses DCT to pass limits through characteristic functions
- Every time you "differentiate under the integral sign" you're invoking DCT

In [ ]:
# --- Numerical demonstration of MCT ---
print("=== Monotone Convergence Theorem Demo ===\n")

# f_n(x) = x^2 * (1 - e^{-nx})  on [0, 2]
# f_n increases to f(x) = x^2

x = np.linspace(0.001, 2, 1000)
true_integral, _ = integrate.quad(lambda t: t**2, 0, 2)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ns_mct = [1, 2, 5, 10, 50]
integrals_mct = []
for n in ns_mct:
    fn = x**2 * (1 - np.exp(-n * x))
    val, _ = integrate.quad(lambda t: t**2 * (1 - np.exp(-n * t)), 0, 2)
    integrals_mct.append(val)
    axes[0].plot(x, fn, label=f'n={n}')

axes[0].plot(x, x**2, 'k--', lw=2, label=r'$f(x)=x^2$ (limit)')
axes[0].set_title('MCT: $f_n \\uparrow f$ pointwise')
axes[0].set_xlabel('x')
axes[0].legend()
axes[0].set_ylim(0, 4.5)

print("MCT: f_n(x) = x²(1 - e^{-nx}) ↑ x² on [0,2]")
print(f"  True ∫₀² x² dx = {true_integral:.6f}")
for n, v in zip(ns_mct, integrals_mct):
    print(f"  n={n:3d}: ∫f_n = {v:.6f}  (error = {abs(v - true_integral):.2e})")


In [ ]:
# --- Numerical demonstration of DCT ---
print("\n=== Dominated Convergence Theorem Demo ===\n")

# f_n(x) = sin(x/n) * e^{-x}  on [0, ∞)
# f_n → 0 pointwise, |f_n| ≤ e^{-x} which is integrable (∫=1)

ns_dct = [1, 2, 5, 10, 50, 200]
integrals_dct = []
x_dct = np.linspace(0, 10, 500)

for n in ns_dct:
    val, _ = integrate.quad(lambda t: np.sin(t / n) * np.exp(-t), 0, np.inf)
    integrals_dct.append(val)
    axes[1].plot(x_dct, np.sin(x_dct / n) * np.exp(-x_dct), label=f'n={n}')

axes[1].plot(x_dct, np.exp(-x_dct), 'k--', lw=2, label=r'$g(x)=e^{-x}$ (dominator)')
axes[1].axhline(0, color='gray', lw=0.5)
axes[1].set_title(r'DCT: $f_n \to 0$, $|f_n| \le e^{-x}$')
axes[1].set_xlabel('x')
axes[1].legend()

plt.tight_layout()
plt.savefig('convergence_theorems.png', dpi=100, bbox_inches='tight')
plt.show()

print("DCT: f_n(x) = sin(x/n)·e^{-x} → 0, dominated by g(x)=e^{-x}")
print(f"  True limit ∫f dμ = 0")
for n, v in zip(ns_dct, integrals_dct):
    print(f"  n={n:3d}: ∫f_n = {v:.6f}")
print("\n✓ Both theorems verified: limit of integrals = integral of limit")


# 12. Product Measures and Fubini's Theorem

## Joint Distributions Are Product Measures

For two random variables $(X, Y)$, the joint distribution lives on
$(\mathbb{R}^2,\;\mathcal{B}(\mathbb{R}^2))$.

**Product $\sigma$-algebra:** $\;\mathcal{F}_1 \otimes \mathcal{F}_2$ is generated by
"rectangles" $A_1 \times A_2$.

**Product measure:** $\;(\mu_1 \times \mu_2)(A_1 \times A_2) = \mu_1(A_1)\cdot\mu_2(A_2)$.

## Fubini's Theorem

If $\int |f|\,d(\mu_1\times\mu_2) < \infty$, then

$$\int\!\!\int f\,d(\mu_1\times\mu_2)
\;=\; \int\!\left(\int f\,d\mu_1\right)d\mu_2
\;=\; \int\!\left(\int f\,d\mu_2\right)d\mu_1$$

**You can integrate in either order.** This is what you do every time you compute a marginal:

$$f_X(x) = \int f_{X,Y}(x,y)\,dy \quad\text{← integrate out } y \text{ (Fubini!)}$$

### Independence as a Product Measure

$X \perp Y \;\Longleftrightarrow\; P_{X,Y} = P_X \times P_Y$

Independence literally means the joint measure *factors*.

In [ ]:
# --- Fubini demonstration with a 2D joint distribution ---
print("=== Fubini's Theorem: Joint Distribution → Marginals ===\n")

# Bivariate normal joint density
mu_x, mu_y = 2.0, 3.0
sigma_x, sigma_y = 1.0, 1.5
rho = 0.6

# Create grid
x_grid = np.linspace(-2, 6, 200)
y_grid = np.linspace(-2, 8, 200)
X, Y = np.meshgrid(x_grid, y_grid)
dx = x_grid[1] - x_grid[0]
dy = y_grid[1] - y_grid[0]

# Joint density
z1 = (X - mu_x) / sigma_x
z2 = (Y - mu_y) / sigma_y
joint = (1 / (2 * np.pi * sigma_x * sigma_y * np.sqrt(1 - rho**2))) * \
        np.exp(-1 / (2 * (1 - rho**2)) * (z1**2 - 2 * rho * z1 * z2 + z2**2))

# Fubini: marginal of X = integrate out Y
marginal_x_numerical = joint.sum(axis=0) * dy  # sum over y-axis
marginal_x_true = stats.norm.pdf(x_grid, mu_x, sigma_x)

# Fubini: marginal of Y = integrate out X
marginal_y_numerical = joint.sum(axis=1) * dx  # sum over x-axis
marginal_y_true = stats.norm.pdf(y_grid, mu_y, sigma_y)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Joint density
c = axes[0].contourf(X, Y, joint, levels=20, cmap='viridis')
axes[0].set_title(r'Joint $f_{X,Y}(x,y)$')
axes[0].set_xlabel('x'); axes[0].set_ylabel('y')
plt.colorbar(c, ax=axes[0])

# Marginal X
axes[1].plot(x_grid, marginal_x_numerical, 'b-', lw=2, label='Numerical (Fubini)')
axes[1].plot(x_grid, marginal_x_true, 'r--', lw=2, label='True N(2,1)')
axes[1].set_title(r'$f_X(x) = \int f_{X,Y}\,dy$')
axes[1].legend()

# Marginal Y
axes[2].plot(y_grid, marginal_y_numerical, 'b-', lw=2, label='Numerical (Fubini)')
axes[2].plot(y_grid, marginal_y_true, 'r--', lw=2, label='True N(3,1.5)')
axes[2].set_title(r'$f_Y(y) = \int f_{X,Y}\,dx$')
axes[2].legend()

plt.tight_layout()
plt.savefig('fubini_marginals.png', dpi=100, bbox_inches='tight')
plt.show()

# Verify: double integral = 1
total_mass = joint.sum() * dx * dy
print(f"∫∫ f(x,y) dx dy ≈ {total_mass:.6f}  (should be 1)")
print(f"Max |marginal_X error| = {np.max(np.abs(marginal_x_numerical - marginal_x_true)):.2e}")
print(f"Max |marginal_Y error| = {np.max(np.abs(marginal_y_numerical - marginal_y_true)):.2e}")
print("\n✓ Fubini verified: integrate in either order → same marginals")


In [ ]:
# --- Independence as product measure ---
print("=== Independence ⟺ Joint = Product of Marginals ===\n")

# Test on Titanic: are sex and pclass independent?
ct = pd.crosstab(titanic['sex'], titanic['pclass'], normalize=True)
marginal_sex = pd.crosstab(titanic['sex'], titanic['pclass'], normalize=True).sum(axis=1)
marginal_class = pd.crosstab(titanic['sex'], titanic['pclass'], normalize=True).sum(axis=0)

# Product of marginals (what we'd see if independent)
product = np.outer(marginal_sex.values, marginal_class.values)
product_df = pd.DataFrame(product, index=marginal_sex.index, columns=marginal_class.index)

print("Joint P(sex, pclass):")
print(ct.round(4))
print("\nProduct of marginals P(sex)×P(pclass):")
print(product_df.round(4))
print(f"\nMax |Joint - Product| = {np.max(np.abs(ct.values - product)):.4f}")
print("If ≈ 0 → independent; if large → dependent")


# 13. The Radon-Nikodym Theorem — Where Densities Come From

## Absolute Continuity

$P \ll \mu$ ("$P$ is absolutely continuous w.r.t. $\mu$") means:

$$\mu(A) = 0 \;\Longrightarrow\; P(A) = 0$$

*$P$ doesn't put mass where $\mu$ sees nothing.*

## The Theorem

If $P \ll \mu$, there exists a measurable function $f = \dfrac{dP}{d\mu} \ge 0$ such that

$$P(A) = \int_A f\,d\mu \quad\text{for all measurable } A$$

This $f$ is the **Radon-Nikodym derivative** — it IS the density!

| Setting | $\mu$ | $dP/d\mu$ | Name |
|---|---|---|---|
| Continuous RV | Lebesgue measure | $f(x)$ | **PDF** |
| Discrete RV | Counting measure | $p(x)$ | **PMF** |
| Model comparison | $P_0$ (null model) | $dP_1/dP_0$ | **Likelihood ratio** |

### Bayes' Theorem as Radon-Nikodym

$$\frac{dP(\theta|x)}{d\pi(\theta)} = \frac{f(x|\theta)}{m(x)}$$

The posterior density IS a Radon-Nikodym derivative!

> 💡 **Physicist's Intuition:** The Radon-Nikodym derivative is like a Jacobian —
> it's the "stretching factor" when you change from one measure to another,
> just as the Jacobian accounts for stretching when changing coordinates.

In [ ]:
# --- PDF as Radon-Nikodym derivative dP/d(Lebesgue) ---
print("=== Radon-Nikodym: Density = dP/dμ ===\n")

# Fit a normal to Housing MedHouseVal
vals = housing['MedHouseVal'].values
mu_hat, sigma_hat = vals.mean(), vals.std()

x_rn = np.linspace(0, 6, 500)
# The PDF IS the R-N derivative dP/d(Lebesgue)
rn_derivative = stats.norm.pdf(x_rn, mu_hat, sigma_hat)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(vals, bins=50, density=True, alpha=0.5, label='Empirical (histogram)')
axes[0].plot(x_rn, rn_derivative, 'r-', lw=2, label=r'$dP/d\lambda$ (fitted normal PDF)')
axes[0].set_title('PDF = Radon-Nikodym Derivative $dP/d\\lambda$')
axes[0].set_xlabel('MedHouseVal')
axes[0].legend()

# Verify: P([a,b]) = ∫_a^b f dλ
a, b = 1.5, 3.0
prob_integral, _ = integrate.quad(stats.norm.pdf, a, b, args=(mu_hat, sigma_hat))
prob_empirical = np.mean((vals >= a) & (vals <= b))
print(f"P(MedHouseVal ∈ [{a}, {b}]):")
print(f"  Via ∫ (dP/dλ) dλ = {prob_integral:.4f}")
print(f"  Via empirical     = {prob_empirical:.4f}")


In [ ]:
# --- Likelihood ratios as dP₁/dP₀ ---
print("\n=== Likelihood Ratios = dP₁/dP₀ ===\n")

# Two hypotheses for Titanic fares:
# H₀: fare ~ Exponential(λ₀)  (null)
# H₁: fare ~ Exponential(λ₁)  (alternative)
fares_clean = titanic['fare'].dropna()
fares_clean = fares_clean[fares_clean > 0]

lambda_0 = 1 / 40.0   # null: mean fare = 40
lambda_1 = 1 / fares_clean.mean()  # MLE

x_lr = np.linspace(0.1, 200, 500)
f0 = stats.expon.pdf(x_lr, scale=1/lambda_0)
f1 = stats.expon.pdf(x_lr, scale=1/lambda_1)

# Likelihood ratio = dP₁/dP₀ = f₁/f₀
lr = f1 / (f0 + 1e-30)

axes[1].plot(x_lr, lr, 'g-', lw=2)
axes[1].axhline(1, color='gray', ls='--', alpha=0.5)
axes[1].set_title(r'Likelihood Ratio $dP_1/dP_0$')
axes[1].set_xlabel('fare')
axes[1].set_ylabel(r'$\Lambda(x) = f_1(x)/f_0(x)$')
axes[1].set_ylim(0, 5)

plt.tight_layout()
plt.savefig('radon_nikodym.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"H₀: fare ~ Exp(mean=40),  λ₀ = {lambda_0:.4f}")
print(f"H₁: fare ~ Exp(mean={1/lambda_1:.1f}), λ₁ = {lambda_1:.4f}")

# Log-likelihood ratio for the observed data
log_lr = np.sum(np.log(stats.expon.pdf(fares_clean, scale=1/lambda_1) + 1e-30) -
                np.log(stats.expon.pdf(fares_clean, scale=1/lambda_0) + 1e-30))
print(f"Log-likelihood ratio: {log_lr:.2f}  ({'favours H₁' if log_lr > 0 else 'favours H₀'})")


# 14. Expectation, Variance, and Moments — The Measure-Theoretic Way

## All the Same Integral

$$E[X] = \int X\,dP = \int x\,dP_X = \int x\,f(x)\,dx$$

These are **literally the same number** computed w.r.t. different measures.

## LOTUS (Law of the Unconscious Statistician)

$$E[g(X)] = \int g(x)\,f(x)\,dx$$

You don't need the distribution of $g(X)$ — just integrate $g$ against the distribution of $X$.

## Variance

$$\text{Var}(X) = E[(X-\mu)^2] = \int (x - \mu)^2\,dP_X$$

## Conditional Expectation: $E[Y|X]$

This is a **random variable** — a function of $X$, not a number!

- **Tower property:** $E[E[Y|X]] = E[Y]$
- **Best predictor:** $E[Y|X]$ minimises $E[(Y - g(X))^2]$ over all measurable $g$
- **Regression function:** $E[Y|X=x]$ IS the regression curve

In [ ]:
# --- Expectations and LOTUS ---
print("=== Expectation: All the Same Integral ===\n")

# Titanic: E[age] computed three equivalent ways
ages = titanic['age'].dropna().values

# Way 1: sample mean = ∫x dPₙ
e_age_empirical = ages.mean()

# Way 2: LOTUS with identity g(x)=x, using histogram as density
hist_counts, bin_edges = np.histogram(ages, bins=50, density=True)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
bin_width = bin_edges[1] - bin_edges[0]
e_age_lotus = np.sum(bin_centers * hist_counts * bin_width)  # ∫x f(x) dx

# Way 3: fit parametric and compute
mu_age, sigma_age = stats.norm.fit(ages)
e_age_parametric = mu_age  # E[X] = μ for normal

print(f"E[age] via empirical measure:   {e_age_empirical:.4f}")
print(f"E[age] via LOTUS (histogram):   {e_age_lotus:.4f}")
print(f"E[age] via fitted Normal(μ,σ):  {e_age_parametric:.4f}")
print("→ All ≈ same value. Different measures, same integral!")

# LOTUS: E[age²]
e_age2_emp = (ages**2).mean()
e_age2_lotus = np.sum(bin_centers**2 * hist_counts * bin_width)
print(f"\nE[age²] empirical:  {e_age2_emp:.2f}")
print(f"E[age²] LOTUS:      {e_age2_lotus:.2f}")
print(f"Var(age) = E[age²]-E[age]² = {e_age2_emp - e_age_empirical**2:.2f}")


In [ ]:
# --- Conditional Expectation E[Y|X] ---
print("=== Conditional Expectation E[Y|X] ===\n")

# Titanic: E[survived | pclass]
cond_exp_survived = titanic.groupby('pclass')['survived'].mean()
print("E[survived | pclass]:")
print(cond_exp_survived.to_string())

# Tower property: E[E[survived|pclass]] = E[survived]
class_counts = titanic.groupby('pclass')['survived'].count()
weights = class_counts / class_counts.sum()
tower = (cond_exp_survived * weights).sum()
print(f"\nTower property: E[E[survived|pclass]] = {tower:.4f}")
print(f"                E[survived]            = {titanic['survived'].mean():.4f}")
print("✓ Tower property verified!")

# Housing: E[MedHouseVal | AveRooms binned]
housing_copy = housing.copy()
housing_copy['rooms_bin'] = pd.cut(housing_copy['AveRooms'], bins=6)
cond_exp_val = housing_copy.groupby('rooms_bin', observed=True)['MedHouseVal'].mean()
print(f"\nHousing: E[MedHouseVal | AveRooms]:")
print(cond_exp_val.to_string())


In [ ]:
# --- LLN: sample mean → E[X] ---
print("\n=== Law of Large Numbers: x̄ₙ → E[X] ===\n")

np.random.seed(0)
true_mean = titanic['fare'].mean()
fare_vals = titanic['fare'].dropna().values

fig, ax = plt.subplots(figsize=(10, 4))
ns = np.arange(1, len(fare_vals) + 1)
running_mean = np.cumsum(fare_vals) / ns
ax.plot(ns, running_mean, 'b-', lw=1, label=r'$\bar{x}_n$')
ax.axhline(true_mean, color='r', ls='--', lw=2, label=f'E[fare] = {true_mean:.2f}')
ax.set_xlabel('n')
ax.set_ylabel('Running mean')
ax.set_title(r'LLN: $\bar{x}_n \to E[X]$ (Titanic fares)')
ax.legend()
plt.tight_layout()
plt.savefig('lln_convergence.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"After n={len(fare_vals)}: x̄ = {running_mean[-1]:.4f}, E[X] = {true_mean:.4f}")


# 15. $L^p$ Spaces — Where Random Variables Live

## Definition

$$L^p(\Omega, \mathcal{F}, \mu) = \left\{ f : \int |f|^p\,d\mu < \infty \right\}$$

| Space | Norm | Meaning |
|---|---|---|
| $L^1$ | $\|f\|_1 = \int|f|\,d\mu$ | Integrable functions |
| $L^2$ | $\|f\|_2 = \sqrt{\int|f|^2\,d\mu}$ | Square-integrable; **Hilbert space** |
| $L^\infty$ | $\|f\|_\infty = \text{ess sup}|f|$ | Bounded functions |

## $L^2$ Is Where the Magic Happens

$L^2$ has an **inner product**: $\;\langle X, Y \rangle = E[XY]$

This makes it a **Hilbert space**, and then:

- **Regression IS projection** in $L^2$: $E[Y|X] = \text{proj}_{L^2(X)} Y$
- **Correlation = cosine** of the angle between centred r.v.s
- **Residuals ⊥ predictions**: $\hat\varepsilon \perp \hat Y$ in $L^2$

## Key Inequalities

**Cauchy-Schwarz:** $|E[XY]| \le \sqrt{E[X^2]\,E[Y^2]}$
→ implies $|\text{Corr}(X,Y)| \le 1$

**Jensen's inequality:** If $g$ is convex, $E[g(X)] \ge g(E[X])$
→ implies $\text{Var}(X) \ge 0$ (take $g(x) = x^2$)

> 💡 **Physicist's Intuition:** $L^2$ IS the Hilbert space of quantum mechanics!
> States $|\psi\rangle \in L^2$, inner product $\langle\phi|\psi\rangle = \int \bar\phi\,\psi\,dx$,
> observables are self-adjoint operators, expectation = $\langle\psi|\hat A|\psi\rangle$.

In [ ]:
# --- Cauchy-Schwarz verification ---
print("=== Cauchy-Schwarz Inequality on Housing Data ===\n")

# Use centred variables (zero-mean)
X_cs = housing['MedInc'].values - housing['MedInc'].mean()
Y_cs = housing['MedHouseVal'].values - housing['MedHouseVal'].mean()

lhs = np.abs(np.mean(X_cs * Y_cs))        # |E[XY]|
rhs = np.sqrt(np.mean(X_cs**2) * np.mean(Y_cs**2))  # √(E[X²]E[Y²])

print(f"|E[XY]|           = {lhs:.6f}")
print(f"√(E[X²]·E[Y²])  = {rhs:.6f}")
print(f"Cauchy-Schwarz:  {lhs:.6f} ≤ {rhs:.6f}  ✓ {'HOLDS' if lhs <= rhs + 1e-10 else 'VIOLATED!'}")

# The ratio IS the correlation (cosine of angle in L²)
corr = lhs / rhs
print(f"\nCorrelation (cosine of angle in L²) = {corr:.6f}")
print(f"scipy corr = {np.corrcoef(X_cs, Y_cs)[0,1]:.6f}")
print(f"|Corr| ≤ 1: {corr:.6f} ≤ 1  ✓")


In [ ]:
# --- Jensen's inequality verification ---
print("\n=== Jensen's Inequality on Housing Data ===\n")

X_jen = housing['MedHouseVal'].values

# g(x) = x² is convex
e_g = np.mean(X_jen**2)         # E[g(X)] = E[X²]
g_e = (np.mean(X_jen))**2       # g(E[X]) = (E[X])²

print("g(x) = x² (convex)")
print(f"E[g(X)] = E[X²]   = {e_g:.6f}")
print(f"g(E[X]) = (E[X])² = {g_e:.6f}")
print(f"Jensen: E[X²] ≥ (E[X])²  →  {e_g:.4f} ≥ {g_e:.4f}  ✓")
print(f"Difference = Var(X) = {e_g - g_e:.6f} ≥ 0  ✓")

# g(x) = e^x is convex
print("\ng(x) = eˣ (convex)")
# Use standardised values to avoid overflow
X_std = (X_jen - X_jen.mean()) / X_jen.std()
e_exp = np.mean(np.exp(X_std))
exp_e = np.exp(np.mean(X_std))
print(f"E[eˣ]  = {e_exp:.6f}")
print(f"e^(E[X]) = {exp_e:.6f}")
print(f"Jensen: E[eˣ] ≥ e^(E[X])  →  {e_exp:.4f} ≥ {exp_e:.4f}  ✓")

# g(x) = -log(x) is convex (x > 0)
print("\ng(x) = -log(x) (convex, x > 0)")
X_pos = X_jen[X_jen > 0]
e_neglog = np.mean(-np.log(X_pos))
neglog_e = -np.log(np.mean(X_pos))
print(f"E[-log(X)]  = {e_neglog:.6f}")
print(f"-log(E[X])  = {neglog_e:.6f}")
print(f"Jensen: {e_neglog:.4f} ≥ {neglog_e:.4f}  ✓")


In [ ]:
# --- Regression as projection in L² ---
print("\n=== Regression = Projection in L² ===\n")

from sklearn.linear_model import LinearRegression

X_reg = housing[['MedInc']].values
Y_reg = housing['MedHouseVal'].values

model = LinearRegression().fit(X_reg, Y_reg)
Y_hat = model.predict(X_reg)
residuals = Y_reg - Y_hat

# In L², residuals ⊥ predictions
inner_product = np.mean(residuals * Y_hat)
print(f"⟨residuals, ŷ⟩ = E[ε·ŷ] = {inner_product:.6e}")
print(f"→ ≈ 0, confirming ε ⊥ ŷ in L²")

# Residuals ⊥ X
inner_product_x = np.mean(residuals * X_reg.ravel())
print(f"⟨residuals, X⟩ = E[ε·X] = {inner_product_x:.6e}")
print(f"→ ≈ 0, confirming ε ⊥ X in L²")

print("\n✓ OLS regression = orthogonal projection in L² Hilbert space")


# 16. How Measure Theory Connects to Everyday Statistics

## The Empirical Measure

Given data $x_1, \dots, x_n$, define the **empirical measure**:

$$\hat P_n = \frac{1}{n}\sum_{i=1}^n \delta_{x_i}$$

Then **everything** you compute from data is an integral w.r.t. $\hat P_n$:

| Statistic | Measure-theoretic form |
|---|---|
| Sample mean $\bar x$ | $\int x\,d\hat P_n$ |
| Sample variance $s^2$ | $\int (x-\bar x)^2\,d\hat P_n$ |
| Histogram | Approximate density $d\hat P_n/d\lambda$ |
| Empirical CDF $\hat F_n(t)$ | $\hat P_n((-\infty, t]) = \int \mathbf{1}_{x \le t}\,d\hat P_n$ |
| MLE | $\arg\max_\theta \int \log f_\theta\,d\hat P_n$ |
| KL divergence | $D_{KL}(P\|Q) = \int \log\frac{dP}{dQ}\,dP$ |

## Convergence of Empirical to True

- **LLN:** $\hat P_n \to P$ (the empirical measure converges to the true measure)
- **Glivenko-Cantelli:** $\sup_t |\hat F_n(t) - F(t)| \to 0$ a.s.
- **CLT:** $\sqrt{n}(\hat P_n - P) \to$ Gaussian process

## Regression = Projection in $L^2$

$E[Y|X]$ is the **orthogonal projection** of $Y$ onto $X$-measurable functions.
Residuals $\perp$ predictions — this is the deepest reason why OLS works.

In [ ]:
# --- Empirical measure construction ---
print("=== The Empirical Measure ===\n")

fare_data = titanic['fare'].dropna().values
n = len(fare_data)

# Empirical measure: P_n(A) = (1/n) * #{x_i in A}
# Demonstrate: sample mean = ∫x dP_n
print(f"n = {n} observations")
print(f"Sample mean = ∫x dP̂ₙ = {fare_data.mean():.4f}")
print(f"Sample var  = ∫(x-x̄)² dP̂ₙ = {fare_data.var():.4f}")

# P_n([0, 50]) = fraction of data in [0, 50]
pn_interval = np.mean((fare_data >= 0) & (fare_data <= 50))
print(f"P̂ₙ([0,50]) = {pn_interval:.4f}")


In [ ]:
# --- Empirical CDF and Glivenko-Cantelli ---
print("=== Glivenko-Cantelli: sup|F̂ₙ - F| → 0 ===\n")

np.random.seed(42)

# True distribution: Normal (fitted to Housing MedHouseVal)
mu_gc, sigma_gc = housing['MedHouseVal'].mean(), housing['MedHouseVal'].std()
true_data = housing['MedHouseVal'].values

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: empirical CDFs for increasing n
t_grid = np.linspace(0, 5.5, 500)
true_cdf = stats.norm.cdf(t_grid, mu_gc, sigma_gc)

sample_sizes = [20, 50, 200, 1000, len(true_data)]
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(sample_sizes)))
sup_diffs = []

for ns_i, color in zip(sample_sizes, colors):
    subsample = true_data[:ns_i]
    ecdf_vals = np.array([np.mean(subsample <= t) for t in t_grid])
    fitted_cdf = stats.norm.cdf(t_grid, mu_gc, sigma_gc)
    sup_diff = np.max(np.abs(ecdf_vals - fitted_cdf))
    sup_diffs.append(sup_diff)
    axes[0].plot(t_grid, ecdf_vals, color=color, lw=1.5,
                 label=f'n={ns_i}, sup|Δ|={sup_diff:.3f}')

axes[0].plot(t_grid, true_cdf, 'k--', lw=2, label='True CDF')
axes[0].set_title('Glivenko-Cantelli: $\\hat F_n \\to F$')
axes[0].set_xlabel('x')
axes[0].set_ylabel('CDF')
axes[0].legend(fontsize=8)

# Right: sup|F_n - F| vs n
axes[1].plot(sample_sizes, sup_diffs, 'ro-', markersize=8)
axes[1].set_xlabel('n')
axes[1].set_ylabel(r'$\sup_t |\hat{F}_n(t) - F(t)|$')
axes[1].set_title('Glivenko-Cantelli: Supremum Error → 0')
axes[1].set_xscale('log')

plt.tight_layout()
plt.savefig('glivenko_cantelli.png', dpi=100, bbox_inches='tight')
plt.show()

for ns_i, sd in zip(sample_sizes, sup_diffs):
    print(f"n={ns_i:5d}: sup|F̂ₙ - F| = {sd:.4f}")
print("\n✓ Glivenko-Cantelli: uniform convergence of empirical CDF")


In [ ]:
# --- Mapping table: everyday statistics ↔ measure theory ---
print("=== The Rosetta Stone: Statistics ↔ Measure Theory ===\n")

rosetta = pd.DataFrame({
    'Everyday Concept': [
        'Sample space Ω',
        'Event',
        'Probability P(A)',
        'Random variable X',
        'PDF f(x)',
        'CDF F(x)',
        'Expectation E[X]',
        'Variance Var(X)',
        'Sample mean x̄',
        'Histogram',
        'Empirical CDF F̂ₙ',
        'Independence',
        'Conditional expectation',
        'MLE',
        'Linear regression',
        'Residuals ⊥ predictions',
        'Correlation',
        'LLN',
        'CLT',
        'KL divergence',
    ],
    'Measure Theory': [
        'Measurable space (Ω, F)',
        'Element of σ-algebra F',
        'Measure μ with μ(Ω)=1',
        'Measurable function X: Ω→ℝ',
        'Radon-Nikodym derivative dP/dλ',
        'P((-∞, x])',
        '∫X dP = ∫x dP_X',
        '∫(X-μ)² dP',
        '∫x dP̂ₙ (empirical measure)',
        'Approx dP̂ₙ/dλ',
        'P̂ₙ((-∞, t])',
        'P_{X,Y} = P_X × P_Y (product measure)',
        'E[Y|σ(X)]: projection in L²',
        'argmax_θ ∫log f_θ dP̂ₙ',
        'Projection in L²(P)',
        'Orthogonality in Hilbert space L²',
        'Cosine of angle in L²',
        'P̂ₙ → P (measure convergence)',
        '√n(P̂ₙ - P) → Gaussian process',
        '∫log(dP/dQ) dP',
    ]
})

print(rosetta.to_string(index=False))


# 17. Almost Everywhere / Almost Surely

## Definitions

- **Almost everywhere (a.e.):** A property holds for all $x$ except possibly a set of
  **measure zero** ($\mu(N) = 0$)
- **Almost surely (a.s.):** Same thing with probability measure — holds with **probability 1**

## Why This Matters

For a continuous random variable $X \sim f$:

$$P(X = x) = 0 \quad\text{for every single } x$$

Yet $X$ obviously takes values! There's no contradiction: each point has measure zero,
but uncountable unions of measure-zero sets can have positive measure.

**Functions equal a.e. are "the same"** in $L^p$ spaces. We don't distinguish between
functions that differ only on a null set.

**Strong LLN:** $\bar X_n \to \mu$ **almost surely** — the set of $\omega$ where
convergence fails has probability zero.

In [ ]:
# --- a.s. / a.e. demonstration ---
print("=== Almost Surely / Almost Everywhere ===\n")

# P(X = x) = 0 for continuous X, yet X takes values
np.random.seed(42)
X_cont = np.random.normal(0, 1, size=100_000)
test_values = [0.0, 1.0, -0.5, 0.12345]

print("For X ~ N(0,1), P(X = x) = 0 for any specific x:")
for x in test_values:
    count = np.sum(X_cont == x)
    print(f"  P(X = {x:8.5f}) ≈ {count}/{len(X_cont)} = {count/len(X_cont):.6f}")

print(f"\nBut P(X ∈ [-1, 1]) = {np.mean((X_cont >= -1) & (X_cont <= 1)):.4f} (≈ 0.6827)")
print(f"    P(X ∈ [-2, 2]) = {np.mean((X_cont >= -2) & (X_cont <= 2)):.4f} (≈ 0.9545)")

# Strong LLN demo
print("\n--- Strong LLN: x̄ₙ → μ almost surely ---")
n_paths = 5
fig, ax = plt.subplots(figsize=(10, 4))

for i in range(n_paths):
    draws = np.random.normal(3.0, 2.0, size=5000)
    running = np.cumsum(draws) / np.arange(1, 5001)
    ax.plot(running, lw=0.8, alpha=0.7, label=f'Path {i+1}')

ax.axhline(3.0, color='red', ls='--', lw=2, label='μ = 3.0')
ax.set_xlabel('n')
ax.set_ylabel(r'$\bar{X}_n$')
ax.set_title(r'Strong LLN: $\bar{X}_n \to \mu$ a.s. (every path converges)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('almost_surely_lln.png', dpi=100, bbox_inches='tight')
plt.show()

print("Every sample path converges → convergence holds almost surely")
print("(The set of non-converging ω has P = 0)")


# 18. Summary — The Big Picture

## The Complete Map

```
Measure Theory          →  Probability & Statistics
─────────────────────────────────────────────────────────────
Measurable space (Ω,F)  →  Sample space + events
σ-algebra F             →  "What can we observe?"
Measure μ               →  Assigns sizes to sets
  μ(Ω) = 1             →  Probability measure P
Measurable function     →  Random variable X
∫f dμ                   →  Expectation E[X]
  w.r.t. counting      →  Summation Σ
  w.r.t. Lebesgue      →  Ordinary integral ∫dx
  w.r.t. empirical P̂ₙ  →  Sample average
dP/dμ (Radon-Nikodym)  →  Density function (PDF/PMF)
Product measure         →  Joint distribution
  P_{XY} = P_X × P_Y   →  Independence
Fubini's theorem        →  Compute marginals
L² Hilbert space        →  Square-integrable r.v.s
  Projection in L²     →  Regression / E[Y|X]
  Cauchy-Schwarz        →  |Corr| ≤ 1
  Jensen's inequality   →  Var ≥ 0, convexity results
MCT, DCT               →  Justify limit-integral swaps
  → underpin LLN, CLT
P̂ₙ → P                →  Law of Large Numbers
√n(P̂ₙ - P) → GP       →  Central Limit Theorem
Almost surely           →  With probability 1
```

## The Chain of Ideas

$$\boxed{\text{Sets}} \xrightarrow{\sigma\text{-algebra}}
\boxed{\text{Measurable}} \xrightarrow{\text{measure}}
\boxed{\text{Probability}} \xrightarrow{\text{integral}}
\boxed{\text{Expectation}} \xrightarrow{L^2}
\boxed{\text{Regression}} \xrightarrow{\text{CLT}}
\boxed{\text{Inference}}$$

In [ ]:
# --- Summary table ---
print("=" * 70)
print("  THE COMPLETE MAP: Everyday Statistics ↔ Measure Theory")
print("=" * 70)

summary = pd.DataFrame({
    'Section': [
        '§7-8: σ-algebras & Measures',
        '§9: Random Variables',
        '§10: Integration',
        '§11: Convergence',
        '§12: Product Measures',
        '§13: Radon-Nikodym',
        '§14: Expectation',
        '§15: Lᵖ Spaces',
        '§16: Statistics Link',
        '§17: Almost Surely',
    ],
    'Key Idea': [
        'Events form σ-algebra; P assigns sizes',
        'RV = measurable function; pushforward measure',
        'Sums, integrals, expectations ALL = ∫f dμ',
        'MCT/DCT: when limits pass through integrals',
        'Joints = products; Fubini → marginals',
        'Densities = dP/dμ; likelihoods = dP₁/dP₀',
        'E[X] = ∫X dP; E[Y|X] = best predictor',
        'L² = Hilbert space; regression = projection',
        'Empirical measure P̂ₙ → LLN, CLT, MLE',
        'a.s. = prob 1; P(X=x)=0 for continuous X',
    ],
    'Why It Matters': [
        'Foundation of everything',
        'Links outcomes to numbers',
        'One framework for all computation',
        'Rigorous limit theorems (LLN, CLT)',
        'Independence, joint distributions',
        'Where PDFs/PMFs actually come from',
        'Conditional expectation = regression',
        'Regression = geometry in function space',
        'Theory meets practice',
        'Convergence in the rigorous sense',
    ]
})

print(summary.to_string(index=False))


## What to Study Next

Now that you have the measure-theoretic foundations, these doors are open:

1. **Martingales** — sequences of r.v.s where $E[X_{n+1}|\mathcal{F}_n] = X_n$.
   Fair games, optional stopping theorem, martingale convergence.

2. **Stochastic Processes** — families of r.v.s indexed by time.
   Markov chains, Poisson processes, Gaussian processes.

3. **Brownian Motion** — the continuous-time limit of random walks.
   Foundation of stochastic calculus, Itô's lemma, Black-Scholes.

4. **Stochastic Calculus** — integration w.r.t. Brownian motion.
   SDEs, Itô vs Stratonovich, Feynman-Kac formula.

5. **Ergodic Theory** — time averages = space averages.
   Birkhoff's theorem, mixing, entropy.

> 💡 **Final Physicist's Intuition:**
>
> The entire edifice of measure-theoretic probability is analogous to the
> transition from classical to quantum mechanics:
>
> | Classical → Quantum | Naive → Rigorous Probability |
> |---|---|
> | Restrict to observables | Restrict to measurable sets (σ-algebra) |
> | Assign amplitudes | Assign measures (probabilities) |
> | Compute ⟨ψ|Â|ψ⟩ | Compute E[X] = ∫X dP |
> | Hilbert space L² | L² space of random variables |
> | Projection postulate | Conditional expectation |
> | Superposition principle | Linearity of expectation |
>
> You already had the intuition — now you have the rigour. 🎓